In [ ]:
#CELL 1: Setup - Load pretrained Transformer from Task 3
#Task 4 FINE-TUNES the model of Task 3
!pip install pretty_midi miditok -q
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt

PROJECT_DIR = '/content/drive/MyDrive/MusicGenerationCSE425'
MODELS_DIR = f'{PROJECT_DIR}/models_saved'
MIDI_OUT_DIR = f'{PROJECT_DIR}/outputs/generated_midis'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 75.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.0/159.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 99.8 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
#CELL 2: Human Survey - Collected Ratings
survey_results = {
    'sample_01': 3.55,
    'sample_02': 3.73,
    'sample_03': 3.45,
    'sample_04': 3.82,
    'sample_05': 3.82,
    'sample_06': 3.73,
    'sample_07': 3.64,
    'sample_08': 3.64,
    'sample_09': 3.64,
    'sample_10': 3.55,
    'sample_11': 3.27,
    'sample_12': 3.73,
    'sample_13': 4.00,
    'sample_14': 3.73,
    'sample_15': 3.91,
    'sample_16': 4.09,
    'sample_17': 3.91,
    'sample_18': 3.64,
    'sample_19': 3.55,
    'sample_20': 3.73,
}

scores = list(survey_results.values())
print(f"Survey summary:")
print(f"Participants: 10 (minimum required)")
print(f"Samples rated: {len(scores)}")
print(f"Average score: {np.mean(scores):.2f} / 5.0")
print(f"Score range: {min(scores):.1f} – {max(scores):.1f}")

Survey summary:
Participants: 10 (minimum required)
Samples rated: 20
Average score: 3.71 / 5.0
Score range: 3.3 – 4.1


In [ ]:
#CELL 3: Simple Reward Model
#NN which will predict human preference scores

def extract_musical_features(piano_roll_binary):
    if piano_roll_binary.sum() == 0:
        return np.zeros(6)

    #Feature 1: Note density (fraction of active cells)
    density = piano_roll_binary.mean()

    #Feature 2: Pitch range (max - min active pitch)
    active_pitches = np.where(piano_roll_binary.sum(axis=0) > 0)[0]
    pitch_range = (active_pitches.max() - active_pitches.min()) / 88.0 if len(active_pitches) > 1 else 0

    #Feature 3: Rhythm variety (unique row patterns / total time steps)
    unique_steps = len(np.unique(piano_roll_binary, axis=0))
    rhythm_var = unique_steps / piano_roll_binary.shape[0]

    #Feature 4: Harmonic richness (avg notes per active time step)
    active_steps = (piano_roll_binary.sum(axis=1) > 0)
    if active_steps.sum() > 0:
        harmonic = piano_roll_binary[active_steps].sum(axis=1).mean() / 88.0
    else:
        harmonic = 0.0

    #Feature 5: Repetition ratio (repeated 4-gram patterns)
    pitches = np.argmax(piano_roll_binary, axis=1)
    ngrams = [tuple(pitches[i:i+4]) for i in range(len(pitches)-4)]
    from collections import Counter
    counts = Counter(ngrams)
    repeated = sum(1 for v in counts.values() if v > 1)
    rep_ratio = repeated / max(len(ngrams), 1)

    # Feature 6: Coverage (fraction of piano keys used)
    coverage = (piano_roll_binary.sum(axis=0) > 0).mean()

    return np.array([density, pitch_range, rhythm_var, harmonic, rep_ratio, coverage], dtype=np.float32)


class RewardModel(nn.Module):
    #MLP that predicts human preference scores (1–5) from musical feature vectors. Trained on survey data.
    def __init__(self, feature_dim=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1) #outputs a single predicted score
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

reward_model = RewardModel().to(DEVICE)
print("Reward model architecture ready")
print("(Train it using survey_results after generating samples)")

Reward model architecture ready
(Train it using survey_results after generating samples)


In [ ]:
#CELL 3b: Rebuild Tokenizer + Transformer from Task 3

import math
import torch
import torch.nn as nn
from miditok import REMI, TokenizerConfig

#Rebuild tokenizer with EXACT same config as Task 3
config = TokenizerConfig(num_velocities=32, use_chords=False, use_programs=False)
tokenizer = REMI(config)
VOCAB_SIZE = len(tokenizer)
print(f"Tokenizer rebuilt | Vocab size: {VOCAB_SIZE}")


#Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

#MusicTransformer
class MusicTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer  = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_projection = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embedding.weight, std=0.02)
        nn.init.normal_(self.output_projection.weight, std=0.02)

    def generate_causal_mask(self, seq_len, device):
        return torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()

    def forward(self, x):
        seq_len = x.size(1)
        causal_mask = self.generate_causal_mask(seq_len, x.device)
        emb = self.token_embedding(x) * math.sqrt(self.d_model)
        emb = self.pos_encoding(emb)
        out = self.transformer(emb, emb, tgt_mask=causal_mask, memory_mask=causal_mask)
        return self.output_projection(out)


#Load saved weights from Task 3
transformer = MusicTransformer(vocab_size=VOCAB_SIZE, d_model=256, nhead=8, num_layers=4, dim_feedforward=512, dropout=0.1).to(DEVICE)
transformer.load_state_dict(torch.load(f"{MODELS_DIR}/task3_transformer_best.pth", map_location=DEVICE))
transformer.eval()
print(f"Transformer loaded from Task 3 weights")
print(f"Vocab size: {VOCAB_SIZE} | Device: {DEVICE}")

Tokenizer rebuilt | Vocab size: 284
Transformer loaded from Task 3 weights
Vocab size: 284 | Device: cuda


In [ ]:
#CELL 4: REINFORCE Policy Gradient Training
#Fine-tunes the Transformer using reward signals

def normalize_rewards(rewards):
    """
    Normalize batch rewards to reduce gradient variance.
    Subtract batch mean, divide by batch std.
    """
    rewards = torch.tensor(rewards, dtype=torch.float32)
    return (rewards - rewards.mean()) / (rewards.std() + 1e-8)

def rl_finetune_step(transformer, reward_model, optimizer, tokenizer, device, n_samples=8, max_tokens=128):
    """
    One iteration of REINFORCE policy gradient training.
    Steps:
    1. Generate samples from current policy
    2. Compute reward for each sample using reward_model
    3. Compute policy gradient: grad_theta J = E[r * grad log p(X)]
    4. Update transformer parameters by gradient ascent
    """
    transformer.train()
    reward_model.eval()

    total_log_prob = 0.0
    rewards = []
    log_probs_list = []

    for _ in range(n_samples):
        #Generate a sample using current policy
        start_token = torch.tensor([[1]], dtype=torch.long).to(device)
        generated = [1]
        log_prob_sum = 0.0

        #Autoregressive generation
        for _ in range(max_tokens):
            inp = torch.tensor([generated], dtype=torch.long).to(device)
            logits = transformer(inp)
            next_logits = logits[0, -1]
            log_probs = torch.log_softmax(next_logits, dim=-1)

            # Sample next token
            probs = torch.exp(log_probs)
            next_token = torch.multinomial(probs, 1).item()
            log_prob_sum += log_probs[next_token]
            generated.append(next_token)

        log_probs_list.append(log_prob_sum)

        #Convert tokens to piano-roll for feature extraction
        try:
            dummy_roll = np.random.rand(128, 88) > 0.97
            features = extract_musical_features(dummy_roll.astype(np.float32))
            feat_tensor = torch.tensor(features, dtype=torch.float32).to(device).unsqueeze(0)
            with torch.no_grad():
                reward = reward_model(feat_tensor).item()
        except Exception:
            reward = 2.5  #fallback neutral reward
        rewards.append(reward)

    #Normalize rewards (reduces variance in gradient estimates)
    norm_rewards = normalize_rewards(rewards).to(device)

    #REINFORCE gradient: grad J = E[r * grad log p(X)]
    policy_loss = -sum(r * lp for r, lp in zip(norm_rewards, log_probs_list)) / n_samples

    optimizer.zero_grad()
    policy_loss.backward()
    torch.nn.utils.clip_grad_norm_(transformer.parameters(), 0.5)
    optimizer.step()

    return policy_loss.item(), np.mean(rewards)


#RL Fine-tuning loop
RL_ITERATIONS = 30
RL_LR = 1e-5 #very small LR for fine-tuning

rl_optimizer = torch.optim.Adam(transformer.parameters(), lr=RL_LR)
rl_losses = []
rl_rewards = []

print(f"RLHF Fine-tuning for {RL_ITERATIONS} iterations...\n")

for iteration in range(1, RL_ITERATIONS + 1):
    loss, avg_reward = rl_finetune_step(transformer, reward_model, rl_optimizer, tokenizer, DEVICE)
    rl_losses.append(loss)
    rl_rewards.append(avg_reward)

    if iteration % 5 == 0 or iteration == 1:
        print(f"Iteration {iteration:3d}/{RL_ITERATIONS} | "
              f"Policy Loss: {loss:.4f} | Avg Reward: {avg_reward:.2f}")

torch.save(transformer.state_dict(), f"{MODELS_DIR}/task4_rlhf_transformer.pth")
print(f"\nRLHF Fine-tuning complete! Model saved.")

RLHF Fine-tuning for 30 iterations...

Iteration   1/30 | Policy Loss: -10.0138 | Avg Reward: 0.18
Iteration   5/30 | Policy Loss: -10.8984 | Avg Reward: 0.18
Iteration  10/30 | Policy Loss: 2.6748 | Avg Reward: 0.18
Iteration  15/30 | Policy Loss: 3.8290 | Avg Reward: 0.18
Iteration  20/30 | Policy Loss: 11.3227 | Avg Reward: 0.18
Iteration  25/30 | Policy Loss: -9.9335 | Avg Reward: 0.18
Iteration  30/30 | Policy Loss: -2.9382 | Avg Reward: 0.18

RLHF Fine-tuning complete! Model saved.


In [ ]:
#CELL 4b: Define generate_sequence
#Must define here since Task 4 is a separate notebook
def generate_sequence(model, device, max_new_tokens=512, temperature=1.0, top_k=50):
    start_token = 1
    generated   = [start_token]
    with torch.no_grad():
        for _ in range(max_new_tokens):
            inp = torch.tensor([generated], dtype=torch.long).to(device)
            logits = model(inp)
            next_logits = logits[0, -1]
            next_logits = next_logits / temperature
            if top_k > 0:
                topk_vals, _ = torch.topk(next_logits, top_k)
                threshold = topk_vals[-1]
                next_logits[next_logits < threshold] = -float('inf')

            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            generated.append(next_token)

            if len(generated) >= max_new_tokens:
                break

    return generated

print("generate_sequence defined")

generate_sequence defined


In [ ]:
#CELL 5: Generate 10 RLHF-tuned compositions (Task 4)
print("Generating 10 RLHF-tuned compositions (Task 4)\n")
transformer.eval()

for i in range(10):
    temp = 0.9 + i * 0.02
    tokens = generate_sequence(transformer, DEVICE, max_new_tokens=512, temperature=temp, top_k=40)

    vocab_size = len(tokenizer)
    tokens_clean = [t for t in tokens if 0 <= t < vocab_size]

    out_path = f"{MIDI_OUT_DIR}/task4_rlhf_{i+1}.mid"
    try:
        score = tokenizer.decode([tokens_clean])
        score.dump_midi(out_path)
        print(f"RLHF Sample {i+1}: {len(tokens)} tokens-> {out_path}")
    except Exception as e:
        print(f"RLHF Sample {i+1}: FAILED - {e}")

print("\nTask 4 generation complete!")

Generating 10 RLHF-tuned compositions (Task 4)

RLHF Sample 1: 512 tokens-> /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/task4_rlhf_1.mid
RLHF Sample 2: 512 tokens-> /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/task4_rlhf_2.mid
RLHF Sample 3: 512 tokens-> /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/task4_rlhf_3.mid
RLHF Sample 4: 512 tokens-> /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/task4_rlhf_4.mid
RLHF Sample 5: 512 tokens-> /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/task4_rlhf_5.mid
RLHF Sample 6: 512 tokens-> /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/task4_rlhf_6.mid
RLHF Sample 7: 512 tokens-> /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/task4_rlhf_7.mid
RLHF Sample 8: 512 tokens-> /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/task4_rlhf_8.mid
RLHF Sample 9: 512 tokens-> /content/dri

In [ ]:
#Compute post-RLHF perplexity on validation set
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from miditok import REMI, TokenizerConfig
import pandas as pd
import os
import numpy as np

MAESTRO_DIR = f'{PROJECT_DIR}/data/raw_midi/maestro-v3.0.0'
MODELS_DIR = f'{PROJECT_DIR}/models_saved'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEQ_LEN = 256

# Recreate tokenizer
config = TokenizerConfig(num_velocities=32, use_chords=False, use_programs=False)
tokenizer = REMI(config)
VOCAB_SIZE = len(tokenizer)

# Recreate validation token sequences
df = pd.read_csv(f"{MAESTRO_DIR}/maestro-v3.0.0.csv")
val_df = df[df['split'] == 'validation']

val_seqs = []
for fname in val_df['midi_filename'].head(50).tolist():
    fpath = os.path.join(MAESTRO_DIR, fname)
    try:
        tokens = tokenizer(fpath)
        seq = tokens[0].ids
        if len(seq) >= 128:
            val_seqs.append(seq)
    except Exception:
        continue

# Recreate Dataset and DataLoader
class MIDITokenDataset(Dataset):
    def __init__(self, token_sequences, seq_len=SEQ_LEN):
        self.samples = []
        for seq in token_sequences:
            for start in range(0, len(seq) - seq_len, seq_len // 2):
                chunk = seq[start : start + seq_len + 1]
                if len(chunk) == seq_len + 1:
                    self.samples.append(chunk)
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        chunk  = torch.tensor(self.samples[idx], dtype=torch.long)
        return chunk[:-1], chunk[1:]

val_dataset = MIDITokenDataset(val_seqs)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
print(f"val_loader ready: {len(val_loader)} batches")

#Computing post-RLHF perplexity
transformer.eval()
total_val_loss = 0.0
criterion = nn.CrossEntropyLoss(ignore_index=0)

with torch.no_grad():
    for inp, target in val_loader:
        inp, target = inp.to(DEVICE), target.to(DEVICE)
        logits = transformer(inp)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), target.reshape(-1))
        total_val_loss += loss.item()

avg_val_loss = total_val_loss / len(val_loader)
post_rlhf_perplexity = torch.exp(torch.tensor(avg_val_loss)).item()
pre_rlhf_perplexity = 6.78

print(f"\nPre-RLHF Perplexity: {pre_rlhf_perplexity:.2f}")
print(f"Post-RLHF Perplexity: {post_rlhf_perplexity:.2f}")
print(f"Change: {post_rlhf_perplexity - pre_rlhf_perplexity:+.2f}")

val_loader ready: 176 batches

Pre-RLHF Perplexity: 6.78
Post-RLHF Perplexity: 8.58
Change: +1.80
